# 📱 Transcriber (Mobile version)

Denna notebook är anpassad för **Carnets** på iPhone/iPad.

## Instruktioner:
1. Kör cell 1 första gången för att installera paket
2. Klistra in YouTube-URL i cell 2
3. Kör cell 2 för att ladda ner transkript
4. (Valfritt) Kör cell 3 för sammanfattning med Claude AI

In [ ]:
# === CELL 1: Installera paket (kör endast första gången) ===
!pip install youtube_transcript_api anthropic python-dotenv

In [1]:
# === CELL 2: Ladda ner transkript ===

import re
import datetime
import urllib.request
import os
from youtube_transcript_api import YouTubeTranscriptApi

# Fråga efter URL
URL = input("YouTube URL: ").strip()

if not URL:
    raise ValueError("❌ Ingen URL angiven!")

# Extrahera video-ID
vid_match = re.search(r"([a-zA-Z0-9_-]{11})", URL)
if not vid_match:
    raise ValueError("❌ Kunde inte hitta video-ID i URL:en")
vid = vid_match.group(1)
print(f"🔍 Video ID: {vid}")

# Hämta metadata från YouTube-sidan
print("📡 Hämtar metadata...")
html = urllib.request.urlopen(f"https://www.youtube.com/watch?v={vid}").read().decode()

# Extrahera titel
title_match = re.search(r"<title>(.+?) - YouTube</title>", html)
title_original = title_match.group(1) if title_match else "Unknown Title"
title_clean = re.sub(r"[^a-zA-Z0-9åäöÅÄÖ ]", "", title_original).replace(" ", "_")[:50]

# Extrahera kanal
channel_match = re.search(r'link itemprop="name" content="([^"]+)"', html)
channel_original = channel_match.group(1) if channel_match else "Unknown Channel"
channel_clean = re.sub(r"[^a-zA-Z0-9åäöÅÄÖ ]", "", channel_original).replace(" ", "_")[:30]

# Extrahera uppladdningsdatum
upload_date_match = re.search(r'"uploadDate":"(\d{4}-\d{2}-\d{2})"', html)
upload_date = upload_date_match.group(1) if upload_date_match else "Unknown"

# Extrahera beskrivning
description_match = re.search(r'"shortDescription":"(.*?)"(?:,"isCrawlable")', html)
if description_match:
    description = description_match.group(1).encode().decode('unicode_escape')
else:
    description = "No description available"

print(f"📺 Titel: {title_original}")
print(f"👤 Kanal: {channel_original}")
print(f"📅 Datum: {upload_date}")

# Hämta transkript
print("📜 Hämtar transkript...")
ytt_api = YouTubeTranscriptApi()
transcript_data = ytt_api.fetch(vid)
transcript_text = " ".join([segment.text for segment in transcript_data])
print(f"📏 Längd: {len(transcript_text):,} tecken")

# Skapa header
now = datetime.datetime.now().strftime("%Y%m%d_%H%M")
header = f"""This file is a transcript from YouTube
Channel: {channel_original}
Title: {title_original}
Upload date: {upload_date}
URL: https://www.youtube.com/watch?v={vid}

Description:
{description}

---

"""

# Spara till fil
output_folder = "transcripts"
os.makedirs(output_folder, exist_ok=True)

filename = f"{now}_{channel_clean}_-_{title_clean}.txt"
filepath = os.path.join(output_folder, filename)

with open(filepath, "w", encoding="utf-8") as f:
    f.write(header + transcript_text)

print(f"\n✅ Transkript sparat!")
print(f"📁 Fil: {filepath}")

# Spara variabler för nästa cell
TRANSCRIPT_CONTENT = header + transcript_text
FILEPATH = filepath
TITLE = title_original
VID = vid

🔍 Video ID: IHcxWYgyeKI
📡 Hämtar metadata...
📺 Titel: The Worldview You Don&#39;t Know You Have &amp; Why It Matters for AI.
👤 Kanal: Aligned Intelligence
📅 Datum: Unknown
📜 Hämtar transkript...
📏 Längd: 4,922 tecken

✅ Transkript sparat!
📁 Fil: transcripts/20260322_0100_Aligned_Intelligence_-_The_Worldview_You_Don39t_Know_You_Have_amp_Why_It_.txt


In [2]:
# cell 3a

!pip install anthropic python-dotenv


  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.13.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached anyio-4.12.1-py3-none-any.whl (113 kB)
Using cached distro-1.9.0-py3-none-any.whl 

In [3]:
# === CELL 3: Summarize with Claude AI (optional) ===

import anthropic
import os

# Fråga om språk
print("Choose language for the summary:")
print("  sv = Swedish")
print("  en = English")
language_code = input("Language (sv/en): ").strip().lower()

language_prompts = {
    "sv": "Sammanfatta följande YouTube-transkript på svenska. Inkludera:\n- Huvudpoänger och nyckelinsikter\n- Viktiga argument eller påståenden\n- Eventuella slutsatser",
    "en": "Summarize the following YouTube transcript in English. Include:\n- Main points and key insights\n- Important arguments or claims\n- Any conclusions"
}

# Default till engelska om ogiltigt val
if language_code not in language_prompts:
    print(f"⚠️ Unknown language '{language_code}', using English")
    language_code = "en"

prompt_template = language_prompts[language_code]
print(f"✅ Language selected: {language_code}")

# Försök ladda API-nyckel från .env först
try:
    from dotenv import load_dotenv
    load_dotenv()
except:
    pass

# Om ingen API-nyckel hittades, fråga användaren
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("\n🔑 No API key found in .env file")
    api_key = input("Enter your Anthropic API key (or press Enter to skip): ").strip()
    if api_key:
        os.environ["ANTHROPIC_API_KEY"] = api_key
    else:
        print("❌ No API key provided. Cannot summarize.")

# Kör sammanfattning om API-nyckel finns
if os.environ.get("ANTHROPIC_API_KEY"):
    print(f"\n📏 Transcript length: {len(TRANSCRIPT_CONTENT):,} characters")
    
    # Trunkera om för långt
    content = TRANSCRIPT_CONTENT
    MAX_CHARS = 150000
    if len(content) > MAX_CHARS:
        print(f"⚠️ Truncating to {MAX_CHARS:,} characters")
        content = content[:MAX_CHARS] + "\n\n[...truncated...]"
    
    print("🤖 Sending to Claude...")
    print("   (may take 10-30 seconds)\n")
    
    client = anthropic.Anthropic()
    
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        messages=[
            {
                "role": "user",
                "content": f"{prompt_template}\n\nTranscript:\n{content}"
            }
        ]
    )
    
    summary = message.content[0].text
    
    print("=" * 50)
    print("📝 SUMMARY")
    print("=" * 50)
    print(summary)
    
    # Spara sammanfattning
    summary_filepath = FILEPATH.replace(".txt", f"_summary_{language_code}.txt")
    with open(summary_filepath, "w", encoding="utf-8") as f:
        f.write(f"Summary of: {TITLE}\n")
        f.write(f"Source: https://www.youtube.com/watch?v={VID}\n")
        f.write(f"Language: {language_code}\n\n")
        f.write(summary)
    
    print(f"\n✅ Summary saved: {summary_filepath}")

Choose language for the summary:
  sv = Swedish
  en = English
⚠️ Unknown language '', using English
✅ Language selected: en

🔑 No API key found in .env file
❌ No API key provided. Cannot summarize.
